# Train RF-DETR-S on the cleaned weightlifting-plates split (Kaggle GPU)

Rebuilds the cleaned dataset from the pinned Roboflow export and the committed
`data/manifest.csv`, fine-tunes RF-DETR-S, and saves everything the evaluation needs as
notebook output. No image is re-hosted anywhere: the manifest decides which images are kept
and which split each one goes to.

**Notebook settings before *Run all*:**

1. *Accelerator*: **GPU T4** (one GPU is used; a second one is left idle).
2. *Internet*: **on** (pip, GitHub, Roboflow, pretrained weights).
3. *Add-ons → Secrets*: **`ROBOFLOW_API_KEY`**, attached to this notebook.

**Test discipline.** The best checkpoint is picked on `valid` only. `test` (the held-out
capture date) is scored once, at the end, by the checkpoint `valid` chose. Nothing in this
notebook should be changed because of a test number.

**Output** (`/kaggle/working/`): `results/` (metrics, config, predictions, run record) is
small and gets committed to the repo; `run/checkpoint_best_total.pth` is the trained model.

In [ ]:
# --- Run configuration: everything that defines this run is here -------------------------
REPO_URL = "https://github.com/Prithv122/custom-detector.git"
REPO_REF = "main"  # branch or commit; the resolved SHA is recorded either way
RFDETR_VERSION = "1.11.0"  # pinned: 1.11 drops Roboflow's placeholder class `objects` itself

EXPECTED_SPLIT_SIZES = {"train": 1263, "val": 331, "test": 402}

TRAIN = dict(
    epochs=50,
    batch_size=8,
    grad_accum_steps=2,  # effective batch 16
    lr=1e-4,
    early_stopping=True,
    early_stopping_patience=10,
    run_test=True,  # score `test` once, with the checkpoint chosen on `valid`
    log_per_class_metrics=True,
)
PRED_THRESHOLD = 0.01  # keep low-confidence boxes so mAP / PR curves can be recomputed

## 1. GPU check

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single GPU: no DDP inside a notebook

import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)

## 2. Install and clone

In [ ]:
%pip install -q "rfdetr[train]=={RFDETR_VERSION}" roboflow imagehash

In [ ]:
import sys
from pathlib import Path

REPO = Path("/tmp/custom-detector")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_REF], check=True)
COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
sys.path.insert(0, str(REPO / "src"))
print("repo at", COMMIT)

## 3. Rebuild the cleaned dataset

Same code path as `custom-detector download` + `custom-detector prepare` on a laptop.
`materialize` refuses any export image the manifest doesn't know.

In [ ]:
from kaggle_secrets import UserSecretsClient

from detector.dataset.manifest import read_manifest
from detector.dataset.prepare import OUTPUT_DIRS, download, materialize

EXPORT = Path("/tmp/raw/projektciezary_v10_coco")
DATA = Path("/tmp/plates_v10_clean")

if not EXPORT.exists():
    download(EXPORT, UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

records = read_manifest(REPO / "data" / "manifest.csv")
counts = materialize(EXPORT, records, DATA)
print(counts)
assert counts == EXPECTED_SPLIT_SIZES, counts

In [ ]:
import json

# Every split must hold exactly the images the manifest assigns to it — no more, no fewer.
for split, folder in OUTPUT_DIRS.items():
    coco = json.loads((DATA / folder / "_annotations.coco.json").read_text())
    got = {im.get("extra", {}).get("name", im["file_name"]) for im in coco["images"]}
    want = {r.file for r in records if r.split == split}
    assert got == want, (split, len(got ^ want))
print("prepared splits match data/manifest.csv")

## 4. Fine-tune RF-DETR-S

In [ ]:
import time
from importlib.metadata import version

import torch
from rfdetr import RFDETRSmall

OUT = Path("/kaggle/working/run")
model = RFDETRSmall()
t0 = time.time()
model.train(dataset_dir=str(DATA), output_dir=str(OUT), **TRAIN)
train_minutes = (time.time() - t0) / 60
print(f"training took {train_minutes:.1f} min")

## 5. Predictions on `valid` and `test`

Saved as flat rows so the evaluation (per-class precision/recall, same-colour confusion,
val → test gap) runs on a laptop and in CI, without a GPU. Class names come from the model,
not from category ids.

In [ ]:
import csv

best = OUT / "checkpoint_best_total.pth"
trained = RFDETRSmall(pretrain_weights=str(best))
RESULTS = Path("/kaggle/working/results")
RESULTS.mkdir(parents=True, exist_ok=True)

for split in ("val", "test"):
    folder = DATA / OUTPUT_DIRS[split]
    coco = json.loads((folder / "_annotations.coco.json").read_text())
    rows = []
    for im in coco["images"]:
        det = trained.predict(
            str(folder / im["file_name"]), threshold=PRED_THRESHOLD, include_source_image=False
        )
        original = im.get("extra", {}).get("name", im["file_name"])
        for (x1, y1, x2, y2), score, name in zip(
            det.xyxy, det.confidence, det.data["class_name"], strict=True
        ):
            rows.append(
                [
                    original,
                    name,
                    f"{score:.4f}",
                    f"{x1:.1f}",
                    f"{y1:.1f}",
                    f"{x2 - x1:.1f}",
                    f"{y2 - y1:.1f}",
                ]
            )
    with (RESULTS / f"predictions_{split}.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "class_name", "confidence", "x", "y", "w", "h"])
        w.writerows(rows)
    print(split, len(coco["images"]), "images,", len(rows), "boxes")

## 6. Run record

In [ ]:
import shutil

for name in ("metrics.csv", "training_config.json"):
    if (OUT / name).exists():
        shutil.copy2(OUT / name, RESULTS / name)

record = {
    "commit": COMMIT,
    "rfdetr": version("rfdetr"),
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "train_minutes": round(train_minutes, 1),
    "split_sizes": counts,
    "train_config": TRAIN,
    "pred_threshold": PRED_THRESHOLD,
}
(RESULTS / "run_record.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
print(sorted(p.name for p in RESULTS.iterdir()))